# ⚛️ Módulo 4: Optimización Cuántica Variacional (QUBO y QAOA)
### Marswalk 3D — NASA Space Apps Challenge

---

## 1. ¿Por qué Computación Cuántica en Rutas Planetarias?
En un grafo discreto clásico ($A^*$), el algoritmo busca nodo a nodo de forma codiciosa en función de costos locales. Esto suele provocar que la ruta "zigzaguee" subiendo y bajando pequeñas terrazas para recortar metros euclidianos.

En biomecánica espacial, un astronauta o rover prefiere una ruta **ergonómica**: contornear curvas de nivel (**isoclinas**) con **baja varianza de pendiente**, aunque la distancia total sea ligeramente mayor.

Formulamos la navegación como un problema de optimización combinatoria en un **Hamiltoniano de Ising / QUBO** resuelto con **QAOA (Quantum Approximate Optimization Algorithm)**.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.dem_pipeline import DEMPipeline
from src.physiology import PhysiologicalModel
from src.qaoa_router import QAOACorridorRouter

data_dir = repo_root / "data" / "jezero_test"
dem_pipe = DEMPipeline(data_dir)
physio = PhysiologicalModel(data_dir / "config_fisio.json")

qaoa_router = QAOACorridorRouter(dem_pipe, physio, p_layers=4)
print("Planificador Cuántico QAOA inicializado (p = 4 capas variacionales)")


## 2. El Corredor Topológico Cuántico

Definimos un corredor adaptativo entre el punto de Inicio $\mathbf{A}$ y Meta $\mathbf{B}$:
- Eje longitudinal $\hat{u} = (\mathbf{B} - \mathbf{A}) / \|\mathbf{B} - \mathbf{A}\|$.
- Eje transversal $\hat{v} = (-\hat{u}_z, \hat{u}_x)$.
- $S$ tajadas longitudinales y $K$ estados discretos transversales por tajada.

Un camino es una configuración de espines/qubits que selecciona exactamente un estado $k$ por cada rebanada $s$ (restricción *one-hot*):
$$x_{s, k} \in \{0, 1\}, \quad \sum_{k=0}^{K-1} x_{s, k} = 1 \quad orall s$$


In [ ]:
start_point = (-4500.0, 2800.0)
goal_point  = (2200.0, -1800.0)

corridor = qaoa_router.generate_corridor_graph(start_point, goal_point, num_slices=20, states_per_slice=11)
slices = corridor["slices"]

print(f"Corredor cuántico generado: {corridor['num_slices']} rebanadas × {corridor['states_per_slice']} estados transversales")
print(f"Distancia en línea recta: {corridor['direct_dist']/1000:.2f} km")


## 3. Hamiltoniano de Costo ($H_C$) y Mezclador ($H_B$)

El Hamiltoniano de Costo codifica el gasto metabólico, la penalización no lineal por escarpes y la suavidad de curvatura:

$$\hat{H}_C = \sum_{s=1}^{S} \sum_{i, j} J_{s, i, j} \hat{Z}_{s-1, i} \hat{Z}_{s, j} + \sum_{s, i} h_{s, i} \hat{Z}_{s, i}$$

Donde los acoplamientos $J_{s, i, j}$ penalizan:
1. El consumo de oxígeno $E_{O_2}(na, nb)$.
2. La pendiente excesiva: $(1 + (	heta / 25^\circ)^2 \cdot 1.35)$.
3. Los saltos laterales bruscos $(\Delta 	ext{offset})^2$, favoreciendo el contorno continuo de isoclinas.

El **Hamiltoniano Mezclador Transversal** genera superposiciones y transiciones cuánticas:
$$\hat{H}_B = \sum_{s, k} \hat{X}_{s, k}$$


## 4. Circuito Cuántico QAOA con Qiskit

El estado variacional cuántico $|\psi(oldsymbol{\gamma}, oldsymbol{eta})angle$ se genera alternando $p$ capas de evolución unitaria:
$$|\psi(oldsymbol{\gamma}, oldsymbol{eta})angle = e^{-i eta_p \hat{H}_B} e^{-i \gamma_p \hat{H}_C} \cdots e^{-i eta_1 \hat{H}_B} e^{-i \gamma_1 \hat{H}_C} |+angle^{\otimes N}$$


In [ ]:
# Construir y visualizar el circuito variacional QAOA con Qiskit
circuit = qaoa_router.build_qiskit_qaoa_circuit(num_qubits=5, p=2)
if circuit:
    print(circuit.draw(output="text"))
else:
    print("Qiskit no disponible; utilizando simulación variacional matricial.")


## 5. Ejecución de la Optimización Cuántica de la Ruta


In [ ]:
route_qaoa = qaoa_router.plan_route(start_point, goal_point, num_slices=28, states_per_slice=21)
mq = route_qaoa["metrics"]

print("--- Métricas de Ruta QAOA (Cuántico-Inspirada) ---")
print(f"Distancia Total:       {mq['distance_m']/1000:.2f} km")
print(f"Oxígeno Consumido:     {mq['oxygen_l']:.1f} Litros")
print(f"Pendiente Máxima:      {mq['max_slope_deg']:.1f}°")
print(f"Varianza de Pendiente: {mq['slope_variance']:.2f}")
print(f"Puntaje de Riesgo:     {mq['risk_score']:.2f} / 1.00")
